In [1]:
from pathlib import Path
import json
import joblib

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

PROJECT_ROOT = Path("..").resolve()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = PROCESSED_DIR / "manual_labels_with_layout_features_v1.csv"

In [2]:
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print(df.columns.tolist())

df.head()

Shape: (175, 35)
['document_name', 'page_number', 'char_count', 'word_count', 'avg_word_length', 'printable_char_ratio', 'whitespace_ratio', 'alphabetic_ratio', 'digit_ratio', 'symbol_ratio', 'text_block_count', 'image_count', 'page_width', 'page_height', 'page_area', 'chars_per_page_area', 'words_per_page_area', 'text_preview', 'label', 'layout_page_width', 'layout_page_height', 'layout_page_area', 'layout_text_block_count', 'layout_image_count', 'total_text_area', 'avg_text_block_area', 'largest_text_block_area', 'total_image_area', 'avg_image_area', 'largest_image_area', 'text_area_ratio', 'image_area_ratio', 'largest_image_area_ratio', 'largest_text_block_ratio', 'text_to_image_area_ratio']


,document_name,page_number,char_count,word_count,avg_word_length,printable_char_ratio,whitespace_ratio,alphabetic_ratio,digit_ratio,symbol_ratio,...,avg_text_block_area,largest_text_block_area,total_image_area,avg_image_area,largest_image_area,text_area_ratio,image_area_ratio,largest_image_area_ratio,largest_text_block_ratio,text_to_image_area_ratio
0,anime_ir_report_80.pdf,1,2182,304,5.953947,0.999542,0.129239,0.807058,0.021082,0.042621,...,3220.193826,6583.045756,0.0,0.0,0.0,0.237862,0.0,0.0,0.013142,0.237862
1,anime_ir_report_80.pdf,2,2032,244,6.102459,0.973917,0.183563,0.701772,0.028543,0.086122,...,4636.197420,61616.470045,0.0,0.0,0.0,0.259156,0.0,0.0,0.123009,0.259156
2,anime_ir_report_80.pdf,3,1671,227,5.744493,0.989228,0.144225,0.742071,0.027528,0.086176,...,2775.924715,15624.390303,0.0,0.0,0.0,0.188420,0.0,0.0,0.031192,0.188420
3,anime_ir_report_80.pdf,4,2090,270,5.570370,0.961244,0.154545,0.679426,0.025359,0.140670,...,8682.899916,61043.886425,0.0,0.0,0.0,0.312017,0.0,0.0,0.121866,0.312017
4,anime_ir_report_80.pdf,5,2203,318,5.449686,0.994553,0.121198,0.714934,0.055379,0.108488,...,3215.418997,22626.932874,0.0,0.0,0.0,0.237509,0.0,0.0,0.045172,0.237509


In [3]:
df["label"] = df["label"].astype(int)

LABEL_NAME_MAP = {
    0: "GOOD_TEXT",
    1: "WEAK_TEXT",
    2: "SCANNED_IMAGE",
    3: "CORRUPTED_TEXT",
    4: "MIXED_CONTENT"
}

OCR_POLICY = {
    0: "NO",
    1: "YES",
    2: "YES",
    3: "YES",
    4: "MAYBE"
}

df["label_name"] = df["label"].map(LABEL_NAME_MAP)
df["ocr_required"] = df["label"].map(OCR_POLICY)

print(df["label_name"].value_counts())
print()
print(df["ocr_required"].value_counts())

label_name
GOOD_TEXT         122
MIXED_CONTENT      22
WEAK_TEXT          11
CORRUPTED_TEXT     11
SCANNED_IMAGE       9
Name: count, dtype: int64

ocr_required
NO       122
YES       31
MAYBE     22
Name: count, dtype: int64


In [4]:
MAX_PAGES_PER_DOC = 10
parts = []

for doc_name, group in df.groupby("document_name"):
    sample = group.sample(
        n=min(len(group), MAX_PAGES_PER_DOC),
        random_state=42
    ).copy()

    sample["document_name"] = doc_name
    parts.append(sample)

balanced_df = pd.concat(parts, ignore_index=True)

print("Balanced shape:", balanced_df.shape)
print()
print(balanced_df["document_name"].value_counts())
print()
print(balanced_df["ocr_required"].value_counts())

Balanced shape: (80, 37)

document_name
CSP544_Final_Report_1.pdf                     10
CryptoPipeline_Manual_UPDATED.pdf             10
Higgs_boson.pdf                               10
anime_ir_report_80.pdf                        10
OSNA_Assignment2[1].pdf                        9
Community areas in Chicago - Wikipedia.pdf     8
Community_Area_Analysis_A_C.pdf                8
application I-765.pdf                          7
OPT I-20 Signed.pdf                            3
Passport.pdf                                   2
Receipt Notice.pdf                             2
Resume General 1.2.0.pdf                       1
Name: count, dtype: int64

ocr_required
NO       43
YES      26
MAYBE    11
Name: count, dtype: int64


In [7]:
FEATURE_COLUMNS = [
    "char_count",
    "word_count",
    "avg_word_length",
    "printable_char_ratio",
    "whitespace_ratio",
    "alphabetic_ratio",
    "digit_ratio",
    "symbol_ratio",
    "text_block_count",
    "image_count",
    "page_width",
    "page_height",
    "page_area",
    "chars_per_page_area",
    "words_per_page_area",
    "layout_text_block_count",
    "layout_image_count",
    "total_text_area",
    "avg_text_block_area",
    "largest_text_block_area",
    "total_image_area",
    "avg_image_area",
    "largest_image_area",
    "text_area_ratio",
    "image_area_ratio",
    "largest_image_area_ratio",
    "largest_text_block_ratio",
    "text_to_image_area_ratio",
]

In [8]:
balanced_df[FEATURE_COLUMNS].isna().sum().sort_values(ascending=False).head(20)

char_count                 0
word_count                 0
avg_word_length            0
printable_char_ratio       0
whitespace_ratio           0
alphabetic_ratio           0
digit_ratio                0
symbol_ratio               0
text_block_count           0
image_count                0
page_width                 0
page_height                0
page_area                  0
chars_per_page_area        0
words_per_page_area        0
layout_text_block_count    0
layout_image_count         0
total_text_area            0
avg_text_block_area        0
largest_text_block_area    0
dtype: int64

In [9]:
balanced_df[FEATURE_COLUMNS] = balanced_df[FEATURE_COLUMNS].fillna(0)

In [ ]:
train_df = balanced_df[
    balanced_df["ocr_required"].isin(["YES", "NO"])
].copy()

test_doc_names = [
    "Community_Area_Analysis_A_C.pdf",
    "Higgs_boson.pdf",
    "application I-765.pdf",
    "anime_ir_report_80.pdf"
]

test_mask = train_df["document_name"].isin(test_doc_names)

train_part = train_df[~test_mask].copy()
test_part = train_df[test_mask].copy()

X_train = train_part[FEATURE_COLUMNS]
y_train = train_part["ocr_required"]

X_test = test_part[FEATURE_COLUMNS]
y_test = test_part["ocr_required"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain distribution:")
print(y_train.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())

print("\nTest documents:")
print(test_part["document_name"].unique())

Train shape: (51, 28)
Test shape: (18, 28)

Train distribution:
ocr_required
NO     34
YES    17
Name: count, dtype: int64

Test distribution:
ocr_required
YES    9
NO     9
Name: count, dtype: int64

Test documents:
<StringArray>
['Community_Area_Analysis_A_C.pdf', 'Higgs_boson.pdf',
 'application I-765.pdf']
Length: 3, dtype: str
